In [3]:
# ===============================================
# SUPERVISED MODELS COMPARISON + METRICS + FEATURE IMPORTANCE
# ===============================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                            mean_absolute_error, mean_squared_error)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

# LOAD DATA
df = pd.read_csv("reviews_sentiment_analysis.csv", encoding="utf-8")
print("Dataset loaded:", df.shape)

# PREPARE FEATURES & TARGET
target_col = "final_sentiment"
text_col = "comments"
numeric_features = ["rating_mean"]

df = df.dropna(subset=[text_col, target_col])
df = df[df[text_col].str.strip() != ""]
df[numeric_features] = df[numeric_features].apply(pd.to_numeric, errors='coerce').fillna(0)

# SPLIT
X_text = df[text_col]
y = df[target_col]
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

# FEATURES
X_num = df[numeric_features]
X_train_num = X_num.loc[X_train_text.index]
X_test_num = X_num.loc[X_test_text.index]

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2), min_df=5)
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)

X_train_final = hstack([X_train_tfidf, X_train_num_scaled])
X_test_final = hstack([X_test_tfidf, X_test_num_scaled])

print(f"Features: {X_train_final.shape[1]} (Text+Numeric)")

# MODELS
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    "Linear SVM": LinearSVC(max_iter=2000, class_weight="balanced", random_state=42)
}

results = []
feature_names = list(tfidf.get_feature_names_out()) + numeric_features

print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80)

le = LabelEncoder()
for name, model in models.items():
    if name == "Random Forest":
        model.fit(X_train_final.toarray(), y_train)
        y_pred = model.predict(X_test_final.toarray())
    else:
        model.fit(X_train_final, y_train)
        y_pred = model.predict(X_test_final)
    
    # METRICS
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    # Convert to numeric for MAE/RMSE (encode labels)
    # le = LabelEncoder()
    y_test_num = le.fit_transform(y_test)
    y_pred_num = le.transform(y_pred)
    mae = mean_absolute_error(y_test_num, y_pred_num)
    rmse = np.sqrt(mean_squared_error(y_test_num, y_pred_num))
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'F1-Macro': f1_macro,
        'F1-Weighted': f1_weighted,
        'MAE': mae,
        'RMSE': rmse
    })
    
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.3f} | F1-Macro: {f1_macro:.3f} | F1-Weighted: {f1_weighted:.3f}")
    print(f"  MAE: {mae:.3f} | RMSE: {rmse:.3f}")
    print(classification_report(y_test, y_pred))

# RESULTS TABLE
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("FINAL METRICS COMPARISON")
print("="*80)
print(results_df.round(4).to_string(index=False))

best_model = results_df.loc[results_df['F1-Macro'].idxmax()]
print(f"\n🏆 BEST MODEL: {best_model['Model']} (F1-Macro = {best_model['F1-Macro']:.3f})")

# ===============================================
# FEATURE IMPORTANCE ANALYSIS
# ===============================================
print("\n" + "="*80)
print("FEATURE IMPORTANCE")
print("="*80)

# 1. Random Forest Feature Importance
rf = models["Random Forest"]
rf_importance = rf.feature_importances_
top_rf_features = np.argsort(rf_importance)[-15:]

print("\n1. Random Forest - Top 15 Features:")
for i, idx in enumerate(reversed(top_rf_features)):
    print(f"  {i+1:2d}. {feature_names[idx]:25s} {rf_importance[idx]:.4f}")

# 2. SVM Weights (LinearSVC coefficients)
svc = models["Linear SVM"]
svc_abs_weights = np.abs(svc.coef_).mean(axis=0)
top_svc_features = np.argsort(svc_abs_weights)[-15:]

print("\n2. SVM - Top 15 Features (by absolute weight):")
for i, idx in enumerate(reversed(top_svc_features)):
    print(f"  {i+1:2d}. {feature_names[idx]:25s} {svc_abs_weights[idx]:.4f}")

# 3. Per-class feature importance (Logistic Regression)
lr = models["Logistic Regression"]
print("\n3. Logistic Regression - Top words per class:")
for i, cls in enumerate(lr.classes_):
    coefs = lr.coef_[i]
    top_pos = np.argsort(coefs)[-5:]
    top_neg = np.argsort(coefs)[:5]
    print(f"\n  {cls.upper()}:")
    print(f"    Positive: {[feature_names[j] for j in top_pos[::-1]]}")
    print(f"    Negative: {[feature_names[j] for j in top_neg]}")

# SAVE EVERYTHING
results_df.to_csv("model_comparison_metrics.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({
    'feature': feature_names,
    'rf_importance': rf_importance,
    'svm_weight': svc_abs_weights
}).to_csv("feature_importance.csv", index=False, encoding="utf-8-sig")

test_results = pd.DataFrame({
    'professor': df.loc[X_test_text.index, 'professor_name_standard'],
    'comment': X_test_text.values,
    'true': y_test.values,
    'rf_pred': models["Random Forest"].predict(X_test_final.toarray()),
    'svm_pred': models["Linear SVM"].predict(X_test_final)
})
test_results.to_csv("test_predictions.csv", index=False, encoding="utf-8-sig")

print("\n✅ SAVED FILES:")
print("  - model_comparison_metrics.csv")
print("  - feature_importance.csv") 
print("  - test_predictions.csv")
print("✓ All metrics (Accuracy/F1/MAE/RMSE) ✓ RF vs SVM Feature Importance ✓ Ready for report!")


Dataset loaded: (4260, 24)
Features: 3001 (Text+Numeric)

MODEL COMPARISON RESULTS

Logistic Regression:
  Accuracy: 0.921 | F1-Macro: 0.906 | F1-Weighted: 0.922
  MAE: 0.085 | RMSE: 0.310
              precision    recall  f1-score   support

    negative       0.83      0.92      0.87       129
     neutral       0.91      0.87      0.89       286
    positive       0.96      0.95      0.96       436

    accuracy                           0.92       851
   macro avg       0.90      0.92      0.91       851
weighted avg       0.92      0.92      0.92       851


Random Forest:
  Accuracy: 0.737 | F1-Macro: 0.664 | F1-Weighted: 0.713
  MAE: 0.361 | RMSE: 0.746
              precision    recall  f1-score   support

    negative       0.98      0.36      0.52       129
     neutral       1.00      0.51      0.67       286
    positive       0.66      1.00      0.80       436

    accuracy                           0.74       851
   macro avg       0.88      0.62      0.66       851
weig